# SkillPatch — Jupyter Test Notebook
Tests every layer of the pipeline in isolation, then wires them together.

**Upload checklist (same folder as this notebook):**
```
tinyvla_debugger/          ← the whole package folder
skillpatch_test.ipynb      ← this file
models/                    ← put your .gguf file here
    Phi-3-mini-4k-instruct-q4.gguf
```

**Layer order tested:**
1. Compiler (mock → llama_cpp ROCm GPU)
2. Failure Classifier
3. Patch Library
4. Audio Feedback (ElevenLabs TTS)
5. Simulated camera frame / real webcam
6. Simulated audio input / real mic → Whisper
7. Full orchestration — all mocks, happy path
8. Full orchestration — failure → auto-patch → recovery
9. Full orchestration — unrecoverable abort
10. Full orchestration — real llama_cpp compiler end-to-end

## Cell 1 — Install dependencies

In [ ]:
import subprocess, sys

pkgs = [
    'opencv-python-headless',
    'numpy',
    'elevenlabs',
]
for pkg in pkgs:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '--break-system-packages', '-q'],
        check=True
    )
    print(f'  ✓ {pkg}')

# llama-cpp-python must be built with ROCm/HIP support — do this in terminal:
# CMAKE_ARGS="-DGGML_HIPBLAS=on" pip install llama-cpp-python --force-reinstall --break-system-packages

# For real mic transcription (optional):
# subprocess.run([sys.executable, '-m', 'pip', 'install', 'sounddevice', 'scipy',
#                 'openai-whisper', '--break-system-packages', '-q'], check=True)

# Audio playback on Linux node:
# !sudo apt-get install -y mpg123 -q

print('\nAll packages ready.')

## Cell 2 — Path setup & imports

In [ ]:
import sys, os, json, asyncio, logging
import numpy as np

# tinyvla_debugger/ must be in the same directory as this notebook.
sys.path.insert(0, '.')

logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s | %(name)s | %(message)s'
)

from tinyvla_debugger.compiler       import SkillCompiler
from tinyvla_debugger.classifier     import FailureClassifier
from tinyvla_debugger.patch_library  import PatchLibrary
from tinyvla_debugger.audio_feedback import AudioFeedback
from tinyvla_debugger.orchestrator   import (
    Orchestrator, SkillAbortError, MockRobotAPI, MockVLMAPI
)
from tinyvla_debugger import trace_logger

print('✓ All imports OK')

## Cell 3 — Layer 1: Compiler (mock backend — no model needed)

In [ ]:
compiler = SkillCompiler(backend='mock')
print(f'Backend: {compiler.backend}\n')

test_commands = [
    'Put the canned goods on the middle shelf',
    'Refill slot 2 only',
    'Stock slots 1 and 2',
]
for cmd in test_commands:
    skill = compiler.compile(cmd)
    print(f'  "{cmd}"')
    print(f'  → skill_name : {skill["skill_name"]}')
    print(f'  → steps      : {len(skill["steps"])}')
    print(f'  → actions    : {[s["action"] for s in skill["steps"]]}')
    print()

## Cell 4 — Diagnose Phi-3 output (run all 4 tests in order)

In [ ]:
GGUF_PATH = './models/Phi-3-mini-4k-instruct-q4.gguf'

if not os.path.exists(GGUF_PATH):
    print(f'⚠ Model not found at {GGUF_PATH}')
else:
    os.environ['PHI3_GGUF_PATH'] = GGUF_PATH
    debug_compiler = SkillCompiler(backend='llama_cpp')  # uses temperature=0.1 now
    print(f'Backend: {debug_compiler.backend}\n')

    # Bypass _parse_and_validate — see the raw string before extraction
    raw = debug_compiler._engine('Stock the top shelf with the juice boxes')
    print('=== Raw output (repr) ===')
    print(repr(raw))
    print()
    print('=== Raw output (plain) ===')
    print(raw)
    print()

    # Also show what _extract_first_json_object pulls out
    extracted = debug_compiler._extract_first_json_object(raw)
    print('=== Extracted JSON object ===')
    print(extracted)

    # ── ROCm GPU status ──────────────────────────────────────────────────
    # If you see "assigned to device CPU" in the verbose output above,
    # llama-cpp-python was compiled without ROCm/HIP support.
    # Fix in terminal (not notebook):
    #   CMAKE_ARGS="-DGGML_HIPBLAS=on" pip install llama-cpp-python \
    #       --force-reinstall --no-cache-dir --break-system-packages
    # Then restart the kernel and re-run from Cell 2.

## Cell 5 — Layer 1: Compiler (llama_cpp — real Phi-3 inference on ROCm GPU)
Requires Cell 4 to have run successfully (model loads once and is reused).

**Fix applied:** switched from `create_chat_completion` to the raw `llm()` completion
API with the Phi-3 chat template built manually. `create_chat_completion` was not
applying the embedded chat template correctly and produced empty `{}` output.
The assistant turn is now primed with `{` so the model is forced to continue the
JSON immediately rather than generating preamble or nothing.

**Also fixed:** `_parse_and_validate` auto-unwraps a single parent key if
Phi-3 wraps the skill object (e.g. `{"result": {...}}`).

In [ ]:
GGUF_PATH = './models/Phi-3-mini-4k-instruct-q4.gguf'

if not os.path.exists(GGUF_PATH):
    print(f'⚠ Skipping: model not found at {GGUF_PATH}')
else:
    os.environ['PHI3_GGUF_PATH'] = GGUF_PATH
    llm_compiler = SkillCompiler(backend='llama_cpp')
    print(f'Backend: {llm_compiler.backend}\n')

    skill = llm_compiler.compile('Stock the top shelf with the juice boxes')
    print(json.dumps(skill, indent=2))

## Cell 6 — Layer 4a: Failure Classifier

In [ ]:
clf = FailureClassifier()

test_cases = [
    # (action, query, vlm_result, retry, expected_failure_type)
    ('pick_from_box', 'Is an object held securely in the gripper?',         False, False, 'GRASP_FAIL'),
    ('place_slot_1',  'Is there an item standing upright in shelf slot 1?',  False, False, 'PLACEMENT_MISS'),
    ('place_slot_1',  'Is there an item standing upright in shelf slot 1?',  False, True,  'PLACEMENT_COLLISION'),
    ('place_slot_2',  'Is this shelf slot currently empty?',                 True,  False, 'DROP_DURING_TRANSIT'),
]

print(f'{"Action":<20} {"Expected":<22} {"Got":<22} OK?')
print('-' * 72)
all_pass = True
for action, query, result, retry, expected in test_cases:
    r = clf.classify(action, query, result, retry=retry)
    ok = '✓' if r.failure_type == expected else '✗ FAIL'
    if r.failure_type != expected:
        all_pass = False
    print(f'{action:<20} {expected:<22} {r.failure_type:<22} {ok}')

print(f'\n{"All tests passed!" if all_pass else "FAILURES DETECTED"}')

## Cell 7 — Layer 4b: Patch Library

In [ ]:
import tempfile, pathlib

tmp_patches = pathlib.Path(tempfile.mktemp(suffix='.json'))
lib = PatchLibrary(patches_file=str(tmp_patches))

base_params = {
    'z_offset_mm': 0.0, 'speed_scale': 1.0,
    'approach_angle_deg': 0.0, 'gripper_close_force': 0.6, 'retry_count': 2
}

lib.store_patch('stock_middle_shelf', 'GRASP_FAIL', {'z_offset_mm': 5.0})
patch = lib.get_patch('stock_middle_shelf', 'GRASP_FAIL')
patched_params = lib.apply_to_params(base_params, patch)

print('Stored patch  :', patch)
print('Base params   :', base_params)
print('Patched params:', patched_params)
assert patched_params['z_offset_mm'] == 5.0
print('\n✓ PatchLibrary OK')

tmp_patches.unlink(missing_ok=True)

## Cell 8 — Audio Feedback (ElevenLabs TTS)
Three events: success, patch recovery, unrecoverable error.

Mock mode prints `[AudioFeedback MOCK]` if `ELEVENLABS_API_KEY` is not set.
Real mode generates an mp3 and plays it via `mpg123` on the node.

In [ ]:
# ── Mock mode ──────────────────────────────────────────────────────────────
print('=== Mock mode (no API key needed) ===')
audio_mock = AudioFeedback()   # silently mocks if ELEVENLABS_API_KEY not set
audio_mock.speak_success('stock_middle_shelf', steps_executed=4, steps_patched=1)
audio_mock.speak_patch_success('GRASP_FAIL', step_id=1)
audio_mock.speak_error('GRASP_FAIL', step_id=1, skill_name='stock_middle_shelf')

# ── Real ElevenLabs ────────────────────────────────────────────────────────
# Requires: ELEVENLABS_API_KEY set + mpg123 installed on node
#
# os.environ['ELEVENLABS_API_KEY'] = 'sk_...'
# audio_real = AudioFeedback()
# audio_real.speak_success('stock_middle_shelf', steps_executed=4, steps_patched=1)
# audio_real.speak_error('GRASP_FAIL', step_id=1, skill_name='stock_middle_shelf')
#
# Custom voice — find your voice_id at:
# https://api.elevenlabs.io/v1/voices  (GET, xi-api-key: YOUR_KEY)
# audio_real = AudioFeedback(voice_id='YOUR_VOICE_ID_HERE')

print('\n✓ AudioFeedback OK')

## Cell 9 — Simulated camera frame
Returns a synthetic BGR numpy array. Swap for `get_real_frame()` when webcam is live.

In [ ]:
import cv2

def get_simulated_frame(scene='shelf', width=640, height=480) -> np.ndarray:
    scenes = {
        'shelf':     (60,  120, 60),
        'gripper':   (80,  80,  160),
        'empty_box': (160, 160, 160),
    }
    color = scenes.get(scene, (100, 100, 100))
    frame = np.full((height, width, 3), color, dtype=np.uint8)
    cv2.putText(frame, f'SIM: {scene}', (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    return frame

def get_real_frame(device_index=0) -> np.ndarray:
    """Swap in when USB webcam is connected."""
    cap = cv2.VideoCapture(device_index)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError(f'Camera index {device_index} not available.')
    return frame

# TODO: HARDWARE — swap to get_real_frame() when webcam is connected
frame = get_simulated_frame(scene='shelf')
print(f'Frame shape : {frame.shape}  dtype: {frame.dtype}')
print('\n✓ Camera frame OK')

## Cell 10 — Simulated audio input → NL command
Randomly picks a demo command. Swap for `get_real_audio_command()` when mic is available.

In [ ]:
import random

DEMO_COMMANDS = [
    'Put the canned goods on the middle shelf',
    'Refill slot 2 only',
    'Stock slots 1 and 2',
    'Move the juice boxes to slot 3',
]

def get_simulated_audio_command() -> str:
    return random.choice(DEMO_COMMANDS)

# def get_real_audio_command(record_seconds=4, sample_rate=16000) -> str:
#     """TODO: HARDWARE — uncomment when mic is available on node."""
#     import sounddevice as sd
#     from scipy.io.wavfile import write
#     import whisper, tempfile
#     print(f'Recording {record_seconds}s... speak now!')
#     audio = sd.rec(int(record_seconds * sample_rate),
#                    samplerate=sample_rate, channels=1, dtype='int16')
#     sd.wait()
#     with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
#         write(f.name, sample_rate, audio)
#         result = whisper.load_model('tiny').transcribe(f.name)
#         os.unlink(f.name)
#     return result['text'].strip()

nl_command = get_simulated_audio_command()
# nl_command = get_real_audio_command()  # TODO: HARDWARE

print(f'NL command: "{nl_command}"')
print('\n✓ Audio input OK')

## Cell 11 — Full orchestration: all mocks, happy path
No hardware needed. Fires `speak_success()` at the end.

In [ ]:
import pathlib, tempfile

tmp_dir    = pathlib.Path(tempfile.mkdtemp())
trace_file = tmp_dir / 'trace.jsonl'
patch_file = tmp_dir / 'patches.json'

robot = MockRobotAPI(failure_on_step=-1)   # -1 = no failures
vlm   = MockVLMAPI(fail_step_ids=set())    # always passes
audio = AudioFeedback(enabled=True)        # mocks if ELEVENLABS_API_KEY not set

orc = Orchestrator(
    robot=robot,
    vlm=vlm,
    audio=audio,
    compiler_backend='mock',
    patches_file=str(patch_file),
    trace_file=str(trace_file),
)

result = asyncio.run(orc.run_skill(nl_command))

print(f'\n=== Result ===')
print(f'  Outcome        : {result.outcome}')
print(f'  Skill name     : {result.skill_name}')
print(f'  Steps executed : {result.steps_executed}')
print(f'  Steps patched  : {result.steps_patched}')
print(f'  Total GPU ms   : {result.total_gpu_ms:.1f}')
assert result.outcome == 'success'
print('\n✓ Happy path OK')

## Cell 12 — Full orchestration: failure → auto-patch → recovery
Step 1 fails on first attempt → classifier → patch applied → retry succeeds.
Fires `speak_patch_success()` then `speak_success()`.
Trace shows: `FAIL` → `patched_success` → `skill_complete`.

In [ ]:
tmp_dir2    = pathlib.Path(tempfile.mkdtemp())
trace_file2 = tmp_dir2 / 'trace.jsonl'
patch_file2 = tmp_dir2 / 'patches.json'

robot2 = MockRobotAPI(failure_on_step=1)   # step 1 gripper fails once
vlm2   = MockVLMAPI(fail_step_ids={1})     # step 1 VLM returns False on attempt 1
audio2 = AudioFeedback(enabled=True)

orc2 = Orchestrator(
    robot=robot2,
    vlm=vlm2,
    audio=audio2,
    compiler_backend='mock',
    patches_file=str(patch_file2),
    trace_file=str(trace_file2),
)

result2 = asyncio.run(orc2.run_skill('Put the canned goods on the middle shelf'))

print(f'\n=== Result ===')
print(f'  Outcome        : {result2.outcome}')
print(f'  Steps patched  : {result2.steps_patched}   ← should be ≥ 1')
assert result2.outcome == 'success'
assert result2.steps_patched >= 1
print('\n✓ Patch recovery path OK')

print('\n=== Trace (newest first) ===')
for evt in trace_logger.read_trace(trace_file=trace_file2):
    print(f'  step={evt["step_id"]}  result={evt["result"]:<20}  '
          f'patch={evt["patch_applied"]}  gpu_ms={evt["gpu_latency_ms"]}')

## Cell 13 — Full orchestration: unrecoverable abort
VLM always returns False → both retry attempts fail → `SkillAbortError` raised.
Fires `speak_error()` just before the raise.

In [ ]:
tmp_dir3    = pathlib.Path(tempfile.mkdtemp())
trace_file3 = tmp_dir3 / 'trace.jsonl'
patch_file3 = tmp_dir3 / 'patches.json'

class AlwaysFailVLM:
    """VLM that always returns False — simulates unrecoverable hardware fault."""
    def verify(self, frame, query):
        return False, 95.0

audio3 = AudioFeedback(enabled=True)
orc3 = Orchestrator(
    robot=MockRobotAPI(),
    vlm=AlwaysFailVLM(),
    audio=audio3,
    compiler_backend='mock',
    patches_file=str(patch_file3),
    trace_file=str(trace_file3),
)

try:
    asyncio.run(orc3.run_skill('Refill slot 2 only'))
    print('ERROR: Should have raised SkillAbortError!')
except SkillAbortError as e:
    print(f'✓ SkillAbortError raised as expected')
    print(f'  failure_type : {e.failure_type}')
    print(f'  step_id      : {e.step_id}')
    print(f'  skill_name   : {e.skill_name}')
    print('  → speak_error() fired just before this raise')

## Cell 14 — Full orchestration: real llama_cpp compiler end-to-end
Combines real Phi-3 inference on the ROCm GPU with mock robot + VLM.
Requires Cell 5 to have passed first.

In [ ]:
GGUF_PATH = './models/Phi-3-mini-4k-instruct-q4.gguf'

if not os.path.exists(GGUF_PATH):
    print(f'⚠ Skipping: model not found at {GGUF_PATH}')
else:
    os.environ['PHI3_GGUF_PATH'] = GGUF_PATH

    tmp_dir4    = pathlib.Path(tempfile.mkdtemp())
    trace_file4 = tmp_dir4 / 'trace.jsonl'

    command = get_simulated_audio_command()
    # command = get_real_audio_command()   # TODO: HARDWARE — real mic
    print(f'Command: "{command}"\n')

    orc4 = Orchestrator(
        robot=MockRobotAPI(),
        vlm=MockVLMAPI(),
        audio=AudioFeedback(enabled=True),
        compiler_backend='llama_cpp',
        trace_file=str(trace_file4),
    )

    result4 = asyncio.run(orc4.run_skill(command))
    print(f'Outcome    : {result4.outcome}')
    print(f'Skill name : {result4.skill_name}')
    print(f'Steps      : {result4.steps_executed}')
    print(f'GPU ms     : {result4.total_gpu_ms:.1f}')

    print('\n=== Steps from Phi-3 ===')
    for evt in reversed(trace_logger.read_trace(trace_file=trace_file4)):
        if evt['step_id'] is not None:
            print(f'  step={evt["step_id"]}  action={evt["action"]:<22}  result={evt["result"]}')

## Cell 15 — Inspect trace.jsonl from any run

In [ ]:
# Change trace_file2 to trace_file, trace_file3, or trace_file4 as needed
events = trace_logger.read_trace(trace_file=trace_file2)

print(f'{len(events)} events (newest first):\n')
for e in events:
    print(json.dumps(e, indent=2))
    print()

## Cell 16 — TODO: HARDWARE checklist
Every row below is currently mocked. Replace one at a time as hardware comes online.

In [ ]:
todo = [
    ('Compiler',         'mock → llama_cpp',              'Set GGUF_PATH; confirm ROCm build (Cell 4 debug)'),
    ('VLM verifier',     'MockVLMAPI → vlm_api',          'Sasha: vlm_api.verify() with ROCMExecutionProvider'),
    ('Robot hardware',   'MockRobotAPI → robot_api',      'Diya: robot_api.replay_skill() / apply_patch()'),
    ('Camera frame',     'get_simulated_frame()',          'Swap to get_real_frame(device_index=0)'),
    ('Audio input',      'get_simulated_audio_command()', 'Swap to get_real_audio_command() + Whisper'),
    ('ElevenLabs TTS',   'mock (print-only)',             'Set ELEVENLABS_API_KEY + apt install mpg123'),
]

print(f'{"Layer":<18} {"Current":<35} Action')
print('-' * 90)
for layer, current, action in todo:
    print(f'{layer:<18} {current:<35} {action}')